# Job Listing Prioritization for Career Changers: EDA/SDA 4 (Statistics)

PredictLeads × TripleTen externship · Jacob Colombo

## 0. Purpose

Vets every candidate feature, non-text (EDA_SDA_2) and text (EDA_SDA_3), with statistical evidence, and hands a **modeling-ready table** to `TimeSeries`, `ML_Fit` and `ML_Rank` notebooks.

**The series** (each notebook hands its final table to the next):

| Notebook | Question | Hands off |
|:--|:--|:--|
| EDA_SDA_1 | What's in the full file, how does tech compare with every field, and what's in the scope? (the scope cut, applied first) | Scope table and its descriptions, occupation summary, company activity |
| EDA_SDA_2 | Which scope listings do we keep, and how do they behave? Additional cleaning, non-text checks by role bucket, ghost listings | Scope table, non-text features |
| EDA_SDA_3 | What do the descriptions say? Text features, text rules as labels and opt-in filters (keyword rules, not validated), skills mapped to O*NET (no matching models) | Scope table + text features |
| EDA_SDA_4 | Which features hold up? Statistics, confounders, the adjusted Cox check | Modeling-ready table |
| TimeSeries | When do listings appear and close? | Timing features |
| ML_Fit | Which roles and listings fit a career changer? Keyword → TF-IDF → BGE → RRF, skills-based matching, a trained layer on real career moves | Fit scores, role discovery, the compass |
| ML_Rank | How should listings be ranked? The listing timing model, combined with fit into the priority queue | The priority queue algorithm |

**Standard:** the TripleTen course templates are the baseline (structure, methods, reporting). Every applicable course method is used, and taken further with more advanced techniques where the data calls for them.

### Clarifying questions
- Which features are redundant with each other (collinearity), and which genuinely overlap?
- Which differences between groups are real, and which are large enough to matter to an applicant?
- Do factor effects hold within each other (for example source within company-volume bands), or reverse (Simpson's paradox)?
- **Adjusted check:** do the factors that survived EDA_SDA_2's within-group checks still matter in one **Cox survival regression** that adjusts for all the confounders at once (company volume, source and platform, staffing agency, role bucket, seniority, month first seen, state, company type, has a description), with errors clustered by company? Report hazard ratios with 95% intervals, and check the proportional-hazards assumption.
- How much does clustering by company inflate apparent certainty?
- Do the career-change flags carry signal beyond seniority, company size and source, and which parts of `career_open` hold up?
- Do factor effects differ by role bucket, or hold across them?
- Are the survival assumptions reasonable (censoring, left truncation)?
- **Is each feature stable?** Does its effect hold in every month (ML_Rank trains on April–July and tests on September), between listings from the same company, and in every role bucket?
- **Is each feature safe?** Is it known when a listing first appears (leakage audit), and does its distribution drift with the collection changes (job boards after May, descriptions from June)?

**Data rules:** aggregates only; never `df.head()` or sample rows. Clear outputs before committing.

### Roadmap and methods

| Section | Course standard | Taken further |
|:--|:--|:--|
| 1. Setup | Load EDA_SDA_3's table | Shared helpers in `src/` |
| 2. Descriptive statistics | Per-group count, mean, median, std, variance, min, max, Q1, Q3; histograms and boxplots by group | Robust summaries, log scales for skewed measures |
| 3. Correlation | **Pearson** `.corr()` with the outcome + heatmap, interpreted by direction and strength | **Spearman**, **Cramér's V** (categories), **point-biserial** (flags); **VIF** for multicollinearity |
| 4. Group rates and interactions | `groupby(feature)[outcome].mean()`; two-way `groupby([A, B])` | Stratified comparisons to check confounding and Simpson's paradox |
| 5. Hypothesis tests | H₀ / H₁, α = 0.05, **Welch's t-test**, assumptions (independence, normality via the central limit theorem, equal variance), sample sizes, practical meaning | **Mann–Whitney (Wilcoxon rank-sum)** / **Kruskal–Wallis** for skewed measures, including EDA_SDA_1's and EDA_SDA_2's gaps, **chi-square** for categories, **log-rank** for survival curves, and its Wilcoxon-weighted version (**Gehan–Breslow–Wilcoxon**) when early differences matter; plain Mann–Whitney never on days open, which is censored; **effect sizes**; **Benjamini–Hochberg** correction across many tests |
| 6. Uncertainty | **Bootstrap** confidence intervals (1,000 resamples, 2.5% / 97.5% quantiles) | **Company-clustered** bootstrap, so listings from one company don't count as independent |
| 7. Time structure | `pivot_table` + heatmap (month × occupation group), `pct_change` | Hands off to `TimeSeries` |
| 8. Survival checks | | Kaplan–Meier with confidence intervals, censoring and left-truncation checks, **competing risks** (filled vs. reposted vs. ghost) described |
| 9. In-depth feature analysis (full data) | | **Stability over months** (each feature's effect per month); **within-company comparisons** (listings from the same company, which removes company-level confounding); **effects by role bucket**; a **leakage audit** (one row per feature: known when the listing first appears?); **drift checks** (distributions over time vs. the collection changes); **information beyond the confounders** (mutual information, likelihood-ratio tests). Priority: stability, within-company and leakage first |
| 10. Feature decisions | Document each decision with data | Each kept / dropped / combined feature recorded with its reason, citing sections 3–9 |
| 11. Final table | | Modeling-ready table, saved for `TimeSeries`, `ML_Fit` and `ML_Rank` |
| 12. Conclusion | What was done, what it means, limitations | |

## 1. Setup

In [ ]:
# 1a Libraries and the shared helpers (src/eda_helpers.py: paths, survival curves, tables, charts)
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "eda_helpers.py").exists())
sys.path.insert(0, str(ROOT))  # so `src` can be imported from notebooks/
from src.eda_helpers import (
    BLUE, ORANGE, AQUA, YELLOW, INK, INK_2, MUTED, CLEAN_DIR,
    as_list, bar_h, finish, km_curve, km_median, km_survival, pct, show, surv_at, use_chart_style,
)
use_chart_style()

In [ ]:
# 1b Statistics helpers, and the two tables to join (only the columns this notebook needs; counts and aggregates are shown, never rows)
import pyarrow.parquet as pq
from src import stats_tests as stx
from src.eda_helpers import PART2_TABLE, PART3_TABLE, PART4_TABLE

SCOPE_COLS = ["id", "company_id", "title", "first_seen", "last_seen", "month", "role_bucket", "has_desc", "live", "days_open", "volume_band",
              "source_type", "is_staffing", "likely_ghost", "level", "is_repost", "career_open", "company_type", "posted", "batch_first_seen"]
have = set(pq.read_schema(PART2_TABLE).names)
scope = pd.read_parquet(PART2_TABLE, columns=[c for c in SCOPE_COLS if c in have])
text_have = pq.read_schema(PART3_TABLE).names
text = pd.read_parquet(PART3_TABLE)  # id + text features + candidate blockers; blank where there is no description
jobs = scope.merge(text, on="id", how="left")
jobs["month"] = jobs.month.astype(str)
jobs["closed"] = (~jobs.live).astype(int)  # event: the listing closed; live = still open at the end (censored)
end = pd.to_datetime(jobs.first_seen).max()
jobs["closed7"] = np.where(pd.to_datetime(jobs.first_seen) <= end - pd.Timedelta(days=7), ((jobs.closed == 1) & (jobs.days_open <= 7)).astype(float), np.nan)  # 7-day close, only where 7 days could be observed
for w in (14, 30):  # sensitivity windows: the main rate is the 7-day one
    jobs[f"closed{w}"] = np.where(pd.to_datetime(jobs.first_seen) <= end - pd.Timedelta(days=w), ((jobs.closed == 1) & (jobs.days_open <= w)).astype(float), np.nan)
if "posted" in jobs:  # days from the posting date to first sighting (only for listings with a posting date); also the left-truncation entry in 8b
    jobs["lag_days"] = ((pd.to_datetime(jobs.first_seen, utc=True) - pd.to_datetime(jobs.posted, utc=True)).dt.total_seconds() / 86400).clip(lower=0)
TEXT_NUMERIC = [c for c in text_have if c not in ("id", "degree_level") and str(text[c].dtype) not in ("bool", "boolean")]
TEXT_FLAGS = [c for c in text_have if c != "id" and str(text[c].dtype) in ("bool", "boolean")]
MISSING = [c for c in SCOPE_COLS if c not in have]
show("1b The joined table", pd.DataFrame({"value": {
    "listings": f"{len(jobs):,}", "companies": f"{jobs.company_id.nunique():,}", "closed": f"{jobs.closed.mean():.1%}",
    "text numeric features": len(TEXT_NUMERIC), "text flags": len(TEXT_FLAGS),
    "scope columns not in the table (skipped)": ", ".join(MISSING) or "none"}}))

**Conventions in this notebook.** The outcome is *closing*: `closed` (1 = the listing closed, 0 = still open when collection ended, i.e. censored) with `days_open` as time. `closed7` (closed within 7 days, only for listings first seen early enough to observe 7 days) is the simple rate used in tables, as in EDA_SDA_1 to 3; `closed14` and `closed30` are sensitivity checks, and 14 days is ML_Rank's main window (`CLOSING_DAYS`), so section 4c and 7a show both. Days open is **never** compared with a plain t-test or Mann–Whitney: it is censored, so survival methods (log-rank, Cox) are used. Every test carries an effect size, and p-values across many tests are Benjamini–Hochberg adjusted (section 5).

## 2. Descriptive statistics

Count, mean, median, std, variance, min, Q1, Q3 and max per group, with histograms and boxplots (log scale for skewed measures). Groups: role bucket, source type, company-volume band, month.

In [ ]:
# 2a Course summary table per group for each numeric measure (days open shown for finished listings only, since open ones are censored)
GROUPS = [g for g in ["role_bucket", "source_type", "volume_band", "month"] if g in jobs]
MEASURES = [m for m in ["years_required", "words_clean"] if m in jobs]
for g in GROUPS:
    for m in MEASURES:
        show(f"2a {m} by {g}", stx.describe_by_group(jobs, m, g).round(2))
    show(f"2a Days open by {g} (finished listings only: a lower bound on the true time)", stx.describe_by_group(jobs[jobs.closed == 1], "days_open", g).round(1))

In [ ]:
# 2b Histograms and boxplots by role bucket (log scale for the skewed measures)
fig, axes = plt.subplots(len(MEASURES), 2, figsize=(11, 3.2 * len(MEASURES)), squeeze=False)
for row, m in zip(axes, MEASURES):
    v = jobs[m].dropna()
    row[0].hist(np.log1p(v), bins=40, color=BLUE)
    row[0].set_title(f"{m} (log1p scale)")
    order = sorted(jobs.role_bucket.dropna().unique())
    row[1].boxplot([np.log1p(jobs.loc[jobs.role_bucket == r, m].dropna()) for r in order], tick_labels=order, vert=False)
    row[1].set_title(f"{m} by role bucket (log1p scale)")
finish()

**What the data shows**
- (fill in after the run)

## 3. Correlation

**Pearson** and **Spearman** of each numeric feature with the 7-day close; **Cramér's V** between categorical factors; **point-biserial** for the 0/1 flags; **VIF** for multicollinearity among the numeric features.

In [ ]:
# 3a Pearson and Spearman with the outcome (closed within 7 days), numeric features
corr = stx.correlations(jobs, TEXT_NUMERIC, "closed7")
show("3a Numeric features vs. closing within 7 days", corr.round(3).sort_values("spearman", key=abs, ascending=False))
fig, ax = plt.subplots(figsize=(6, 0.3 * len(corr) + 1))
heatmap_vals = corr[["pearson", "spearman"]].astype(float)
im = ax.imshow(heatmap_vals.values, cmap="RdBu_r", vmin=-0.3, vmax=0.3, aspect="auto")
ax.set_xticks([0, 1], heatmap_vals.columns); ax.set_yticks(range(len(heatmap_vals)), heatmap_vals.index)
fig.colorbar(im); ax.set_title("Correlation with closing within 7 days")
finish()

In [ ]:
# 3b Cramer's V between categorical factors, and point-biserial of each 0/1 flag with the outcome
CATS = [c for c in ["role_bucket", "source_type", "volume_band", "month", "level", "company_type"] if c in jobs]
cv = pd.DataFrame({a: {b: stx.cramers_v(jobs[a], jobs[b]) for b in CATS} for a in CATS})
show("3b Cramer's V between categorical factors (0 = unrelated, 1 = identical)", cv.round(2))
pb = {f: stx.point_biserial(jobs[f].astype(float), jobs.closed7) for f in TEXT_FLAGS if jobs[f].notna().any()}
show("3b Point-biserial of each flag with closing within 7 days", pd.DataFrame(pb, index=["r", "p"]).T.round(4).sort_values("r", key=abs, ascending=False))

In [ ]:
# 3c VIF among the numeric features (a common rule of thumb flags 5 or 10+); features with blanks use their complete rows
sample = jobs[TEXT_NUMERIC].dropna()
sample = sample.sample(min(len(sample), 200_000), random_state=0)
show("3c Variance inflation factors", stx.vif(sample, [c for c in TEXT_NUMERIC if sample[c].nunique() > 1]).round(2).to_frame())

**What the data shows**
- (fill in after the run)

## 4. Group rates, interactions and Simpson's paradox

Rate of closing within 7 days by factor, two-way by role bucket, and for each flag the crude gap against the gap within strata (company volume, source, role bucket). Opposite signs mean Simpson's paradox: the factor looks one way overall and the other within groups.

In [ ]:
# 4a One-way and two-way rates (by role bucket)
for f in [c for c in ["source_type", "volume_band", "level", "month"] if c in jobs]:
    show(f"4a Closing within 7 days by {f}", stx.rate_by(jobs, "closed7", f).round(3))
    show(f"4a ... by role bucket and {f}", stx.rate_by(jobs, "closed7", ["role_bucket", f])["rate"].unstack().round(3))

In [ ]:
# 4b Simpson's paradox check: crude vs. stratum-weighted gap for each flag, within each confounder
CONF = [c for c in ["volume_band", "source_type", "role_bucket", "month", "level"] if c in jobs]
rows = []
for f in TEXT_FLAGS:
    for s in CONF:
        _, summ = stx.stratified_gap(jobs, "closed7", f, s)
        rows.append({"flag": f, "within": s, **summ})
simpson = pd.DataFrame(rows)
show("4b Crude vs. adjusted gap (percentage points of 7-day closing); `reverses` marks a sign flip", simpson.round(4).set_index(["flag", "within"]))
show("4b Flags whose gap reverses within some confounder", simpson[simpson.reverses][["flag", "within"]].set_index("flag") if simpson.reverses.any() else pd.DataFrame({"note": ["none"]}))

**What the data shows**
- (fill in after the run)

### 4c. Sensitivity to the window

Do the rates and flag gaps change if the window is 14 or 30 days instead of 7?

In [ ]:
# 4c Close rate by role bucket at 7, 14 and 30 days, and each flag's pooled gap at each window
show("4c Close rate by role bucket and window", jobs.groupby("role_bucket", observed=True)[["closed7", "closed14", "closed30"]].mean().round(3))
gaps = {f: {w: stx.effect_by_group(jobs, w, f, "role_bucket", min_n=1).attrs["pooled_gap"] for w in ["closed7", "closed14", "closed30"]} for f in TEXT_FLAGS}
show("4c Pooled gap (flag vs. not) by window", pd.DataFrame(gaps).T.round(4))

**What the data shows**
- (fill in after the run)

## 5. Hypothesis tests

H₀: the groups do not differ; H₁: they do; α = 0.05. Assumptions: independence (listings of one company are not independent: section 6 widens the intervals for this), large samples (central limit theorem for Welch), and equal variances not assumed. **Welch's t-test** and **Mann–Whitney** compare measures that are fully observed (words, years); **chi-square** compares categories and flags; **log-rank** (and Gehan–Breslow–Wilcoxon, which weights early differences) compares closing curves because days open is censored. Each test reports an effect size (Cohen's d, rank-biserial, Cramér's V, event-rate ratio), and the p-values are Benjamini–Hochberg adjusted across the whole family. This is where EDA_SDA_1's and EDA_SDA_2's descriptive gaps get significance tests.

In [ ]:
# 5a Welch + Mann-Whitney for each numeric feature: closed within 7 days vs. not
res = []
for f in TEXT_NUMERIC:
    d = jobs[["closed7", f]].dropna()
    a, b = d.loc[d.closed7 == 1, f], d.loc[d.closed7 == 0, f]
    if len(a) > 30 and len(b) > 30:
        res += [{"name": f"{f}: welch", **stx.welch_test(a, b)}, {"name": f"{f}: mann-whitney", **stx.mann_whitney(a, b)}]
t5a = stx.tests_table(res)
show("5a Numeric features, closed in 7 days vs. not", t5a[["n_a", "n_b", "p", "effect", "effect_name", "q_bh", "significant_after_bh"]].round(4))

In [ ]:
# 5b Chi-square for each flag and categorical factor against the 7-day close; log-rank (and Wilcoxon-weighted) for each flag on the full closing curve
res = [{"name": f"{c} x closed7: chi-square", **stx.chi_square(jobs[c], jobs.closed7)} for c in CATS + TEXT_FLAGS if jobs[c].notna().any()]
for f in TEXT_FLAGS:
    d = jobs[["days_open", "closed", f]].dropna()
    a, b = d[d[f].astype(bool)], d[~d[f].astype(bool)]
    if len(a) > 30 and len(b) > 30:
        for w in (False, True):
            res.append({"name": f"{f}: " + ("gehan-breslow-wilcoxon" if w else "log-rank"), **stx.logrank(a.days_open, a.closed, b.days_open, b.closed, weighted=w)})
t5b = stx.tests_table(res)
show("5b Categories and survival curves", t5b[["n_a", "n_b", "p", "effect", "effect_name", "q_bh", "significant_after_bh"]].round(4))

**What the data shows**
- (fill in after the run)

## 6. Uncertainty: company-clustered bootstrap

1,000 resamples of whole companies (2.5% / 97.5% quantiles), against the plain row-level bootstrap. The ratio of widths is how much clustering by company inflates apparent certainty.

In [ ]:
# 6a Bootstrap intervals for the 7-day close rate overall and by role bucket; the plain bootstrap is run on a sample to keep time down
rate = lambda f: f.closed7.mean()
cols = ["company_id", "closed7", "role_bucket"]
base = jobs[cols].dropna(subset=["closed7"])
rows = {}
for name, d in [("all", base), *[(r, g) for r, g in base.groupby("role_bucket", observed=True)]]:
    cl = stx.clustered_bootstrap_rate(d, "company_id", "closed7", n_boot=1000, seed=0)
    pl = stx.plain_bootstrap(d.sample(min(len(d), 200_000), random_state=0), rate, n_boot=1000, seed=0)
    rows[name] = {"rate": cl["estimate"], "clustered_low": cl["ci_low"], "clustered_high": cl["ci_high"], "plain_low": pl["ci_low"], "plain_high": pl["ci_high"],
                  "width_ratio": (cl["ci_high"] - cl["ci_low"]) / max(pl["ci_high"] - pl["ci_low"], 1e-12), "companies": cl["n_clusters"]}
show("6a 7-day close rate with 95% intervals", pd.DataFrame(rows).T.round(4))

**What the data shows**
- (fill in after the run)

## 7. Time structure

Listings by month and role bucket, and the month-to-month change in the 7-day close rate. The detailed timing analysis is `TimeSeries`.

In [ ]:
# 7a Month x role bucket counts and close rates, and the month-to-month change
piv = jobs.pivot_table(index="month", columns="role_bucket", values="closed7", aggfunc="mean", observed=True)
show("7a 7-day close rate by month and role bucket", piv.round(3))
show("7a Month-to-month change in the rate (percentage points)", (piv.diff() * 100).round(1))
cnt = jobs.pivot_table(index="month", columns="role_bucket", values="id", aggfunc="count", observed=True)
show("7a Listings by month and role bucket", cnt)
show("7a Close rate by month at 7, 14 and 30 days (sensitivity)", jobs.groupby("month")[["closed7", "closed14", "closed30"]].mean().round(3))

**What the data shows**
- (fill in after the run)

## 8. Survival checks

Censoring (how many listings are still open and when they were first seen), **left truncation** (listings already open when collection began enter the risk set late; ignoring it overstates survival), and **competing risks** (a listing can end by being filled or turn out a ghost; described here, not modeled).

In [ ]:
# 8a Censoring by role bucket and by month first seen
show("8a Censoring by role bucket", stx.censoring_summary(jobs.days_open, jobs.closed, jobs.role_bucket).round(3))
show("8a Censoring by month", stx.censoring_summary(jobs.days_open, jobs.closed, jobs.month).round(3))

In [ ]:
# 8b Left truncation. A listing whose posting date is earlier than its first sighting was already open when we first saw it, so it
# enters the risk set at that age: ENTRY = lag_days (posting date to first sighting). Only listings with a posting date (about 45%) are used;
# the time axis is then age since posting = lag_days + days_open.
ENTRY_COL = "lag_days"
if ENTRY_COL in jobs:
    d = jobs.loc[jobs.posted.notna() & (jobs.days_open > 0), [ENTRY_COL, "days_open", "closed"]].dropna()
    d = d.assign(age=d[ENTRY_COL] + d.days_open)
    show("8b Survival by age since posting at 7, 14, 30 days: ignoring vs. accounting for left truncation", stx.left_truncation_gap(d.age, d.closed, d[ENTRY_COL]).round(3))
else:
    show("8b Left truncation", pd.DataFrame({"note": ["no posting-date column in the scope table"]}))

In [ ]:
# 8b-2 Sensitivity: drop crawl-artifact listings (batch first sightings) and listings first seen in the opening days of collection
OPENING_DAYS = 7
start = pd.to_datetime(jobs.first_seen).min()
early = pd.to_datetime(jobs.first_seen) < start + pd.Timedelta(days=OPENING_DAYS)
batch = jobs.batch_first_seen.fillna(False).astype(bool) if "batch_first_seen" in jobs else pd.Series(False, index=jobs.index)
rows = {"all listings": jobs, "without batch first sightings": jobs[~batch], f"without the first {OPENING_DAYS} days of collection": jobs[~early], "without either": jobs[~batch & ~early]}
tab = {k: {"listings": len(v), "7-day close rate": v.closed7.mean(), **{f"gap: {f}": stx.effect_by_group(v, "closed7", f, "role_bucket", min_n=1).attrs["pooled_gap"] for f in TEXT_FLAGS}} for k, v in rows.items()}
show("8b-2 Rates and flag gaps with and without the artifacts", pd.DataFrame(tab).T.round(4))

In [ ]:
# 8c Competing risks: share of listings that ended as closed or likely ghost (descriptive; 0 = still open).
# "Ended by being reposted" needs a later-copy column, which the scope table doesn't have (is_repost marks a repost of an
# earlier listing, known at first sighting), so it is not a cause here.
jobs["end_cause"] = np.select([jobs.live, jobs.likely_ghost.fillna(False).astype(bool)], [0, 2], default=1)
show("8c Cause of ending (0 open, 1 closed, 2 likely ghost)", jobs.end_cause.value_counts(normalize=True).sort_index().round(3).to_frame("share"))
d = jobs.loc[jobs.days_open > 0, ["days_open", "end_cause"]]
show("8c Cumulative incidence by cause at 7, 30, 90 days (Aalen-Johansen)", stx.cumulative_incidence(d.days_open, d.end_cause, [1, 2]).round(3))

**What the data shows**
- (fill in after the run)

### 8d. Adjusted check: Cox survival regression

One Cox model with every standard confounder at once (company volume, source, staffing agency, role bucket, seniority, month first seen, company type if available, has a description) plus the features under test, with **company-clustered** standard errors. Hazard ratio above 1 = closes faster. Reported with 95% intervals and a proportional-hazards (Schoenfeld) check. The model is fitted on **all rows** (its hazard ratios are ML_Rank's starting weights) with a small ridge penalizer of 0.01 to keep the fit stable with many dummy columns; with a penalizer the intervals are approximate. The likelihood-ratio tests in 9f use a 300,000-listing sample instead. Which features go in the model is the analyst's call: the lists below are the starting point.

In [ ]:
# 8d Cox regression: confounders + the flags under test
CONFOUNDERS = [c for c in ["volume_band", "source_type", "is_staffing", "role_bucket", "level", "month", "company_type", "has_desc"] if c in jobs]
CAT_CONF = [c for c in CONFOUNDERS if jobs[c].dtype == object or str(jobs[c].dtype) in ("category", "string")]
UNDER_TEST = TEXT_FLAGS  # edit: the features whose adjusted effect is wanted
cox_df = jobs[jobs.days_open > 0].copy()
cox_df[UNDER_TEST] = cox_df[UNDER_TEST].astype(float).fillna(0)  # flags are blank without a description: 0 here, has_desc carries the gap
for c in set(CONFOUNDERS + UNDER_TEST):
    if str(cox_df[c].dtype) in ("bool", "boolean"):
        cox_df[c] = cox_df[c].astype(float)
cph, design, hr = stx.cox_adjusted(cox_df,  # all rows: these hazard ratios become ML_Rank's starting weights
     "days_open", "closed", CONFOUNDERS + UNDER_TEST, cluster="company_id", categorical=CAT_CONF, penalizer=0.01)
show("8d Hazard ratios (above 1 = closes faster) with 95% intervals", hr.round(3))
show("8d Proportional-hazards check (small p = the effect changes over time)", stx.proportional_hazards_check(cph, design).round(4))

**What the data shows**
- (fill in after the run)

## 9. In-depth feature analysis (full data)

Priority: 9a stability over months, 9b within-company comparisons, 9d leakage audit first.

### 9a. Stability over months

Does each flag's effect on the 7-day close hold in every month? ML_Rank trains on April–July and tests on September, so a feature that flips sign across months is a risk.

In [ ]:
# 9a Per-feature stability over months (gap in 7-day close rate, flag vs. not)
stab = stx.stability_over_months(jobs, "closed7", TEXT_FLAGS, "month", min_n=100)
show("9a Stability over months (share_same_sign: months whose gap has the pooled sign)", stab.round(4).sort_values("share_same_sign"))

**What the data shows**
- (fill in after the run)

### 9b. Within-company comparisons

Among listings of the same company, does the flag still predict closing? This removes company-level confounding (a company's size, habits, pay).

In [ ]:
# 9b Within-company vs. crude gap for each flag
wc = pd.DataFrame({f: stx.within_company_effect(jobs, "closed7", f, "company_id") for f in TEXT_FLAGS}).T
show("9b Within-company gap vs. crude gap (companies_used = companies with both kinds of listing)", wc.round(4))

**What the data shows**
- (fill in after the run)

### 9c. Effects by role bucket

In [ ]:
# 9c Gap by role bucket for each flag
by_role = pd.DataFrame({f: stx.effect_by_group(jobs, "closed7", f, "role_bucket", min_n=100)["gap"] for f in TEXT_FLAGS}).T
show("9c Gap in 7-day close rate by role bucket", by_role.round(4))

**What the data shows**
- (fill in after the run)

### 9d. Leakage audit

One row per feature: is it known when the listing first appears? Anything only known later (how long it stayed open, how many times it was reposted, later edits) must not be a model feature. The answers are filled in by hand; `None` means not yet decided.

In [ ]:
# 9d Leakage audit, pre-filled from the rule in STANDARDS.md (only what is known at first sighting). Text flags and has_desc stay
# "pending" until EDA_SDA_2's 3.9 check (is the description the one seen at first sighting?) is done. None = to decide.
PENDING = "pending EDA_SDA_2 3.9 check: description as seen at first sighting?"
audit = {f: (None, PENDING) for f in TEXT_NUMERIC + TEXT_FLAGS + (["has_desc"] if "has_desc" in jobs else [])}
KNOWN = {"level": "seniority is in the title", "source_type": "source is fixed at first sighting", "company_id": "company is known at first sighting",
         "volume_band": "company volume: check it uses only listings up to first sighting", "first_seen": "first_seen is the first sighting itself",
         "month": "month of first_seen", "is_staffing": "company type is known at first sighting", "company_type": "company type is known at first sighting",
         "salary_hourly_usd": "pay is in the listing", "is_repost": "earlier copies only: a repost is known when it appears"}
NOT_ALLOWED = {"last_seen": "STANDARDS.md: never use", "live": "status is only known later", "closed": "the outcome",
               "days_open": "the outcome", "later_reposts": "reposts after the first sighting are in the future"}
audit.update({k: (True, v) for k, v in KNOWN.items() if k in jobs})
audit.update({k: (False, v) for k, v in NOT_ALLOWED.items()})
show("9d Leakage audit (STANDARDS.md: nothing from after the listing first appears)", stx.leakage_audit(audit))

**What the data shows**
- (fill in after the run)

### 9e. Drift checks

Which distributions shift over time, and do they line up with the collection changes (job boards after May, descriptions from June)? Standardized mean difference against the first month, and the Kolmogorov–Smirnov distance.

In [ ]:
# 9e Drift per numeric feature and per flag (std_mean_diff above ~0.2 is a visible shift)
drift = {f: stx.drift_over_time(jobs, f, "month")["std_mean_diff"] for f in TEXT_NUMERIC + TEXT_FLAGS + ["has_desc"] if f in jobs}
show("9e Standardized mean difference vs. the first month", pd.DataFrame(drift).T.round(2))

**What the data shows**
- (fill in after the run)

### 9f. Information beyond the confounders

A likelihood-ratio test: does the feature improve a logistic model of the 7-day close that already holds the confounders? Plus mutual information with the outcome.

In [ ]:
# 9f LR test per flag, on a sample for speed; and mutual information
lr_pool = jobs.dropna(subset=["closed7"])
lr_sample = lr_pool.sample(min(len(lr_pool), 300_000), random_state=0)
for c in set(CONFOUNDERS):
    if str(lr_sample[c].dtype) in ("bool", "boolean"):
        lr_sample[c] = lr_sample[c].astype(float)
lr = {}
for f in TEXT_FLAGS:
    d = lr_sample.dropna(subset=[f])
    if d[f].nunique() > 1:
        conf = [c for c in CONFOUNDERS if c != "has_desc"]  # constant among listings with a description
        lr[f] = stx.information_beyond_confounders(d, "closed7", f, conf, categorical=CAT_CONF)
show("9f Likelihood-ratio test beyond the confounders", pd.DataFrame(lr).T.round(4))
show("9f Mutual information with the 7-day close", stx.mutual_information(lr_sample, "closed7", TEXT_NUMERIC + [f for f in TEXT_FLAGS if lr_sample[f].notna().any()]).round(5).sort_values(ascending=False).to_frame())

**What the data shows**
- (fill in after the run)

## 10. Feature decisions

Each feature kept, dropped or combined, with its reason and the section it comes from. **To be filled in by hand from sections 3–9**; nothing is decided automatically.

In [ ]:
# 10a Decision log: (feature, "kept" | "dropped" | "combined", reason, evidence section). Empty until the analyst decides.
DECISIONS = [
    # ("years_required", "kept", "reason", "5a, 9a"),
]
decisions = stx.decision_log(DECISIONS) if DECISIONS else pd.DataFrame(columns=["decision", "reason", "evidence"])
show("10a Feature decisions", decisions if len(decisions) else pd.DataFrame({"note": ["no decisions recorded yet"]}))

**What the data shows**
- (fill in after the run)

## 11. Final table

One row per scope listing (`id`) with the kept and combined features, plus the identifiers and outcome columns the next notebooks need (`title` for ML_Rank's keyword baseline, `last_seen` for its targets; neither is a feature). Saved as `modeling_ready.parquet` in `data/clean/` (`PART4_TABLE`) for `TimeSeries`, `ML_Fit` and `ML_Rank`; the name is this notebook's choice, since EDA_SDA_3 hands off `text_table.parquet` and the scope table (joined by `id`).

In [ ]:
# 11a Save the modeling-ready table (refuses to save an empty decision log)
KEEP_ALWAYS = ["company_id", "title", "first_seen", "last_seen", "month", "role_bucket", "days_open", "live", "closed"]  # last_seen and title: ML_Rank's targets and keyword baseline, never features
if len(decisions):
    ready = stx.modeling_ready(scope.assign(closed=jobs.closed.values), text, decisions, keep_always=[c for c in KEEP_ALWAYS if c in jobs])
    ready.to_parquet(PART4_TABLE, index=False)
    show("11a The modeling-ready table", pd.DataFrame({"value": {
        "listings": f"{len(ready):,}", "columns": ready.shape[1], "saved": f"{PART4_TABLE.name} ({PART4_TABLE.stat().st_size / 1e9:.2f} GB)"}}))
else:
    show("11a The modeling-ready table", pd.DataFrame({"note": ["not saved: fill in the decisions in 10a first"]}))

## 12. Conclusion

_To write after the run: what was done, what it means, limitations._

**Handoff:** `modeling_ready.parquet` (kept and combined features, joined by `id`) for `TimeSeries`, `ML_Fit` and `ML_Rank`.